# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 6: The Autonomous & Self-Optimizing Ecosystem (Distributed Intelligence)

### Key assumptions
- Vehicles, MFCs, and customers operate as autonomous intelligent agents
- Game-theoretic negotiation protocols enable resource allocation
- Continuous auction mechanisms determine optimal assignments
- Nash equilibrium solutions emerge from distributed decision-making
- No centralized coordination required for system optimization

### Approach (step-by-step)
1. **Agent Definition**: Define autonomous agents for each system component
2. **Utility Functions**: Create utility functions for each agent type
3. **Negotiation Protocol**: Implement auction-based negotiation mechanisms
4. **Distributed Decision-Making**: Enable agent-to-agent interactions
5. **Equilibrium Analysis**: Monitor convergence to Nash equilibrium

### What to look for in the results
- Distributed negotiation outcomes and auction results
- Nash equilibrium achievement and system efficiency
- Agent utility maximization and fairness metrics
- Self-organizing behavior and emergent system properties

### Concrete example (from the source)
Vehicle V-23 negotiates with MFC-7, MFC-12, MFC-18 for package assignments, achieving 94.3% vehicle utilization while maintaining 97.1% on-time delivery.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for Autonomous Ecosystem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import random
import time
from datetime import datetime, timedelta
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
try:
    class AgentType(Enum):
        VEHICLE = "vehicle"
        MFC = "mfc"
        CUSTOMER = "customer"
    
    @dataclass
    class DeliveryPackage:
        """Represents a delivery package in the ecosystem"""
        id: int
        customer_id: int
        origin_mfc: int
        destination: Tuple[float, float]
        size: float  # cubic meters
        priority: int  # 1=low, 2=medium, 3=high
        time_window: Tuple[datetime, datetime]
        reserve_price: float  # Minimum acceptable price
    
    @dataclass
    class Bid:
        """Represents a bid in the auction mechanism"""
        agent_id: int
        agent_type: AgentType
        price: float
        capacity: float
        delivery_time: timedelta
        confidence: float  # Confidence in meeting the bid
    
    class AutonomousAgent:
        """Base class for autonomous agents in the ecosystem"""
        
        def __init__(self, agent_id: int, agent_type: AgentType, location: Tuple[float, float]):
            self.id = agent_id
            self.type = agent_type
            self.location = location
            self.utility = 0.0
            self.resources = {}
            self.history = []
            self.reputation = 0.5  # Initial reputation score
            
        def calculate_utility(self, outcome: Dict) -> float:
            """Calculate utility based on outcome - to be overridden"""
            return 0.0
        
        def make_bid(self, package: DeliveryPackage, context: Dict) -> Optional[Bid]:
            """Make a bid for a package - to be overridden"""
            return None
        
        def update_reputation(self, success: bool, performance: float):
            """Update reputation based on performance"""
            if success:
                self.reputation = min(1.0, self.reputation + 0.05 * performance)
            else:
                self.reputation = max(0.0, self.reputation - 0.1)
    
    class VehicleAgent(AutonomousAgent):
        """Autonomous vehicle agent"""
        
        def __init__(self, agent_id: int, location: Tuple[float, float], capacity: float, 
                     speed: float, vehicle_type: str):
            super().__init__(agent_id, AgentType.VEHICLE, location)
            self.capacity = capacity
            self.speed = speed
            self.vehicle_type = vehicle_type
            self.current_load = 0.0
            self.current_route = []
            self.availability_window = 8.0  # hours
            self.cost_per_km = 0.50  # Operating cost
            
        def calculate_utility(self, outcome: Dict) -> float:
            """Calculate utility for vehicle agent"""
            revenue = outcome.get('revenue', 0)
            distance = outcome.get('distance', 0)
            time_cost = outcome.get('time_cost', 0)
            load_factor = outcome.get('load_factor', 0)
            
            # Utility = Revenue - Costs + Load Factor Bonus
            transport_cost = distance * self.cost_per_km
            time_utility = -time_cost * 10  # Penalty for time
            load_bonus = load_factor * 20  # Bonus for high utilization
            
            utility = revenue - transport_cost + time_utility + load_bonus
            return utility
        
        def make_bid(self, package: DeliveryPackage, context: Dict) -> Optional[Bid]:
            """Make a bid for package delivery"""
            # Check capacity constraints
            if self.current_load + package.size > self.capacity:
                return None
            
            # Calculate distance and time
            distance = self._calculate_distance(package.destination)
            travel_time = timedelta(hours=distance / self.speed)
            
            # Check time window feasibility
            current_time = context.get('current_time', datetime.now())
            earliest_delivery = current_time + travel_time
            
            if earliest_delivery > package.time_window[1]:
                return None  # Cannot meet time window
            
            # Calculate bid price based on costs and market factors
            base_cost = distance * self.cost_per_km
            market_factor = context.get('market_factor', 1.0)
            urgency_factor = 1.0 + (package.priority - 1) * 0.2
            
            # Competitive bidding strategy
            competitors = context.get('competitors', 1)
            competition_factor = max(0.8, 1.0 - competitors * 0.05)
            
            bid_price = base_cost * market_factor * urgency_factor * competition_factor
            
            # Ensure minimum profit margin
            min_price = base_cost * 1.2
            bid_price = max(bid_price, min_price, package.reserve_price)
            
            # Calculate confidence based on reputation and load
            confidence = self.reputation * (1.0 - self.current_load / self.capacity)
            
            return Bid(
                agent_id=self.id,
                agent_type=AgentType.VEHICLE,
                price=bid_price,
                capacity=self.capacity - self.current_load,
                delivery_time=travel_time,
                confidence=confidence
            )
        
        def _calculate_distance(self, destination: Tuple[float, float]) -> float:
            """Calculate distance to destination"""
            return np.sqrt((destination[0] - self.location[0])**2 + 
                          (destination[1] - self.location[1])**2)
    
    class MFCAgent(AutonomousAgent):
        """Autonomous micro-fulfillment center agent"""
        
        def __init__(self, agent_id: int, location: Tuple[float, float], capacity: int):
            super().__init__(agent_id, AgentType.MFC, location)
            self.capacity = capacity
            self.current_packages = []
            self.processing_cost_per_package = 2.0
            self.storage_cost_per_package = 0.5
            
        def calculate_utility(self, outcome: Dict) -> float:
            """Calculate utility for MFC agent"""
            revenue = outcome.get('revenue', 0)
            processing_cost = outcome.get('processing_cost', 0)
            storage_cost = outcome.get('storage_cost', 0)
            throughput = outcome.get('throughput', 0)
            
            # Utility = Revenue - Costs + Throughput Bonus
            total_cost = processing_cost + storage_cost
            throughput_bonus = throughput * 5  # Bonus for high throughput
            
            utility = revenue - total_cost + throughput_bonus
            return utility
        
        def make_bid(self, package: DeliveryPackage, context: Dict) -> Optional[Bid]:
            """Make a bid for package processing"""
            # Check capacity
            if len(self.current_packages) >= self.capacity:
                return None
            
            # Calculate processing costs
            processing_cost = self.processing_cost_per_package
            storage_time = (package.time_window[1] - datetime.now()).total_seconds() / 3600
            storage_cost = storage_time * self.storage_cost_per_package
            
            total_cost = processing_cost + storage_cost
            
            # Calculate service fee
            service_fee = total_cost * 1.5  # 50% margin
            
            # Adjust based on demand and reputation
            demand_factor = context.get('demand_factor', 1.0)
            reputation_bonus = self.reputation * 0.2
            
            final_price = service_fee * demand_factor * (1 + reputation_bonus)
            
            return Bid(
                agent_id=self.id,
                agent_type=AgentType.MFC,
                price=final_price,
                capacity=self.capacity - len(self.current_packages),
                delivery_time=timedelta(hours=1),  # Processing time
                confidence=self.reputation
            )
    
    class AuctionMechanism:
        """Distributed auction mechanism for resource allocation"""
        
        def __init__(self):
            self.auction_history = []
            self.market_prices = {}
            
        def run_auction(self, package: DeliveryPackage, bidders: List[AutonomousAgent], 
                       context: Dict) -> Optional[Tuple[AutonomousAgent, Bid]]:
            """Run auction for package allocation"""
            
            # Collect bids
            bids = []
            for bidder in bidders:
                bid = bidder.make_bid(package, context)
                if bid and bid.price >= package.reserve_price:
                    bids.append((bidder, bid))
            
            if not bids:
                return None
            
            # Sort by price (descending) then by confidence (descending)
            bids.sort(key=lambda x: (x[1].price, x[1].confidence), reverse=True)
            
            # Winner determination
            winner, winning_bid = bids[0]
            
            # Record auction
            auction_record = {
                'package_id': package.id,
                'winner_id': winner.id,
                'winner_type': winner.type.value,
                'winning_price': winning_bid.price,
                'num_bidders': len(bids),
                'timestamp': datetime.now()
            }
            self.auction_history.append(auction_record)
            
            return winner, winning_bid
    
    class DistributedEcosystem:
        """Autonomous self-optimizing ecosystem"""
        
        def __init__(self, city_size: Tuple[float, float] = (100, 100)):
            self.city_size = city_size
            self.vehicles = []
            self.mfcs = []
            self.packages = []
            self.auction = AuctionMechanism()
            self.current_time = datetime.now()
            
            # System metrics
            self.total_utility = 0.0
            self.nash_equilibrium_achieved = False
            self.equilibrium_iterations = 0
            
        def initialize_ecosystem(self, num_vehicles: int = 5, num_mfcs: int = 3):
            """Initialize the autonomous ecosystem"""
            print("Initializing Autonomous Ecosystem...")
            
            # Create vehicle agents
            vehicle_configs = [
                ('van', 12.0, 50.0, 2),
                ('bike', 2.0, 25.0, 2),
                ('drone', 1.0, 40.0, 1)
            ]
            
            vehicle_id = 1
            for v_type, capacity, speed, count in vehicle_configs:
                for _ in range(count):
                    location = (random.uniform(0, self.city_size[0]), 
                               random.uniform(0, self.city_size[1]))
                    vehicle = VehicleAgent(vehicle_id, location, capacity, speed, v_type)
                    self.vehicles.append(vehicle)
                    vehicle_id += 1
            
            # Create MFC agents
            for i in range(num_mfcs):
                location = (random.uniform(20, self.city_size[0]-20), 
                           random.uniform(20, self.city_size[1]-20))
                mfc = MFCAgent(i+1, location, capacity=100)
                self.mfcs.append(mfc)
            
            print(f"✅ Created {len(self.vehicles)} vehicle agents and {len(self.mfcs)} MFC agents")
        
        def generate_packages(self, num_packages: int = 15):
            """Generate delivery packages for auction"""
            for i in range(num_packages):
                # Random customer location
                destination = (random.uniform(0, self.city_size[0]), 
                              random.uniform(0, self.city_size[1]))
                
                # Select random MFC as origin
                origin_mfc = random.choice(self.mfcs).id
                
                # Time window (next 4-8 hours)
                start_time = self.current_time + timedelta(hours=random.uniform(0, 4))
                end_time = start_time + timedelta(hours=random.uniform(2, 6))
                
                package = DeliveryPackage(
                    id=i+1,
                    customer_id=random.randint(1000, 9999),
                    origin_mfc=origin_mfc,
                    destination=destination,
                    size=random.uniform(0.1, 2.0),
                    priority=random.choices([1, 2, 3], weights=[0.7, 0.25, 0.05])[0],
                    time_window=(start_time, end_time),
                    reserve_price=random.uniform(10, 50)
                )
                
                self.packages.append(package)
            
            print(f"📦 Generated {num_packages} packages for auction")
        
        def run_distributed_optimization(self, max_iterations: int = 10) -> Dict:
            """Run distributed optimization through agent negotiations"""
            print("\n🔄 Running Distributed Optimization...")
            
            optimization_results = {
                'iterations': [],
                'total_utility': [],
                'packages_allocated': [],
                'equilibrium_convergence': []
                'auction_results': []
            }
            
            for iteration in range(max_iterations):
                print(f"Iteration {iteration + 1}/{max_iterations}")
                
                # Reset allocations for this iteration
                allocated_packages = 0
                iteration_utility = 0.0
                iteration_auctions = []
                
                # Run auctions for each package
                for package in self.packages:
                    # Create auction context
                    context = {
                        'current_time': self.current_time,
                        'market_factor': 1.0 + random.uniform(-0.1, 0.1),
                        'competitors': len(self.vehicles),
                        'demand_factor': 1.0 + random.uniform(-0.2, 0.2)
                    }
                    
                    # Run auction between vehicles
                    result = self.auction.run_auction(package, self.vehicles, context)
                    
                    if result:
                        winner, winning_bid = result
                        
                        # Update agent state
                        if isinstance(winner, VehicleAgent):
                            winner.current_load += package.size
                            winner.current_route.append(package.id)
                        
                        # Calculate utilities
                        outcome = {
                            'revenue': winning_bid.price,
                            'distance': winner._calculate_distance(package.destination) if isinstance(winner, VehicleAgent) else 0,
                            'time_cost': winning_bid.delivery_time.total_seconds() / 3600,
                            'load_factor': winner.current_load / winner.capacity if isinstance(winner, VehicleAgent) else 0
                        }
                        
                        utility = winner.calculate_utility(outcome)
                        iteration_utility += utility
                        
                        allocated_packages += 1
                        
                        # Record auction result
                        auction_result = {
                            'package_id': package.id,
                            'winner_type': winner.type.value,
                            'winner_id': winner.id,
                            'price': winning_bid.price,
                            'utility': utility
                        }
                        iteration_auctions.append(auction_result)
                
                # Record iteration results
                optimization_results['iterations'].append(iteration + 1)
                optimization_results['total_utility'].append(iteration_utility)
                optimization_results['packages_allocated'].append(allocated_packages)
                optimization_results['auction_results'].append(iteration_auctions)
                
                # Check for equilibrium (convergence)
                if iteration > 0:
                    utility_change = abs(iteration_utility - optimization_results['total_utility'][-2])
                    convergence_threshold = 0.01 * iteration_utility
                    
                    if utility_change < convergence_threshold:
                        optimization_results['equilibrium_convergence'].append(True)
                        if len(optimization_results['equilibrium_convergence']) >= 3:  # 3 consecutive convergences
                            self.nash_equilibrium_achieved = True
                            self.equilibrium_iterations = iteration + 1
                            print(f"✅ Nash equilibrium achieved at iteration {iteration + 1}")
                            break
                    else:
                        optimization_results['equilibrium_convergence'].append(False)
                else:
                    optimization_results['equilibrium_convergence'].append(False)
                
                # Reset agent states for next iteration
                for vehicle in self.vehicles:
                    vehicle.current_load = 0.0
                    vehicle.current_route = []
            
            self.total_utility = optimization_results['total_utility'][-1] if optimization_results['total_utility'] else 0
            
            return optimization_results
    
    print("Autonomous Ecosystem implementation complete")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: invalid syntax. Perhaps you forgot a comma? (<string>, line 310)...]

In [ ]:
try:
    # Initialize and run the Autonomous Ecosystem
    print("=" * 60)
    print("INITIALIZING AUTONOMOUS SELF-OPTIMIZING ECOSYSTEM")
    print("=" * 60)
    
    # Create ecosystem instance
    ecosystem = DistributedEcosystem(city_size=(100, 100))
    
    # Initialize agents
    ecosystem.initialize_ecosystem(num_vehicles=5, num_mfcs=3)
    
    # Generate packages
    ecosystem.generate_packages(num_packages=15)
    
    print(f"\n📊 ECOSYSTEM STATUS:")
    print(f"   City size: {ecosystem.city_size[0]}x{ecosystem.city_size[1]} km")
    print(f"   Vehicle agents: {len(ecosystem.vehicles)}")
    print(f"   MFC agents: {len(ecosystem.mfcs)}")
    print(f"   Packages for allocation: {len(ecosystem.packages)}")
    
    # Display agent details
    print(f"\n🚚 VEHICLE AGENTS:")
    for vehicle in ecosystem.vehicles:
        print(f"   V-{vehicle.id}: {vehicle.vehicle_type}, capacity={vehicle.capacity}m³, speed={vehicle.speed}km/h")
    
    print(f"\n📦 MFC AGENTS:")
    for mfc in ecosystem.mfcs:
        print(f"   MFC-{mfc.id}: location={mfc.location}, capacity={mfc.capacity} packages")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'DistributedEcosystem' is not defined...]

In [ ]:
try:
    # Run distributed optimization
    print("=" * 60)
    print("RUNNING DISTRIBUTED OPTIMIZATION")
    print("=" * 60)
    
    # Run optimization with multiple iterations
    optimization_results = ecosystem.run_distributed_optimization(max_iterations=10)
    
    # Display results
    print(f"\n🎯 OPTIMIZATION RESULTS:")
    print(f"   Total iterations: {len(optimization_results['iterations'])}")
    print(f"   Nash equilibrium achieved: {ecosystem.nash_equilibrium_achieved}")
    if ecosystem.nash_equilibrium_achieved:
        print(f"   Equilibrium iteration: {ecosystem.equilibrium_iterations}")
    
    # Final iteration analysis
    if optimization_results['total_utility']:
        final_utility = optimization_results['total_utility'][-1]
        final_allocation = optimization_results['packages_allocated'][-1]
        
        print(f"\n📈 FINAL PERFORMANCE:")
        print(f"   Total system utility: {final_utility:.2f}")
        print(f"   Packages allocated: {final_allocation}/{len(ecosystem.packages)}")
        print(f"   Allocation rate: {final_allocation/len(ecosystem.packages)*100:.1f}%")
        
        # Calculate efficiency metrics
        if optimization_results['auction_results']:
            final_auctions = optimization_results['auction_results'][-1]
            avg_price = np.mean([a['price'] for a in final_auctions])
            avg_utility = np.mean([a['utility'] for a in final_auctions])
            vehicle_wins = len([a for a in final_auctions if a['winner_type'] == 'vehicle'])
            
            print(f"\n💰 AUCTION METRICS:")
            print(f"   Average auction price: ${avg_price:.2f}")
            print(f"   Average utility per agent: {avg_utility:.2f}")
            print(f"   Vehicle agent wins: {vehicle_wins}/{len(final_auctions)}")
            print(f"   Vehicle win rate: {vehicle_wins/len(final_auctions)*100:.1f}%")
    
    # Show detailed auction results from final iteration
    if optimization_results['auction_results'] and optimization_results['auction_results'][-1]:
        print(f"\n📋 DETAILED AUCTION RESULTS (Final Iteration):")
        final_auctions = optimization_results['auction_results'][-1]
        
        for auction in final_auctions[:5]:  # Show first 5
            print(f"   Package {auction['package_id']}: {auction['winner_type']}-{auction['winner_id']} "
                  f"won at ${auction['price']:.2f} (utility: {auction['utility']:.2f})")
        
        if len(final_auctions) > 5:
            print(f"   ... and {len(final_auctions) - 5} more auctions")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ecosystem' is not defined...]

In [ ]:
try:
    # Demonstrate specific negotiation example from problem statement
    print("=" * 60)
    print("DEMONSTRATION: V-23 NEGOTIATION SCENARIO")
    print("=" * 60)
    
    # Create specific scenario matching problem statement
    print("\n🎭 Setting up V-23 negotiation scenario...")
    
    # Create V-23 vehicle agent
    v23 = VehicleAgent(
        agent_id=23,
        location=(45.2, 32.8),
        capacity=3.2,
        speed=35.0,
        vehicle_type='van'
    )
    v23.current_load = 0.0  # Just completed delivery 1,247
    v23.reputation = 0.85  # Experienced agent
    
    # Create MFC agents
    mfc7 = MFCAgent(agent_id=7, location=(25.5, 40.2), capacity=150)
    mfc12 = MFCAgent(agent_id=12, location=(60.1, 28.7), capacity=120)
    mfc18 = MFCAgent(agent_id=18, location=(35.8, 55.3), capacity=100)
    
    # Set MFC reputations
    mfc7.reputation = 0.90
    mfc12.reputation = 0.75
    mfc18.reputation = 0.80
    
    # Create packages from each MFC
    packages = [
        DeliveryPackage(
            id=1001, customer_id=891, origin_mfc=7,
            destination=(48.3, 35.6), size=2.8, priority=2,
            time_window=(datetime.now() + timedelta(hours=1), datetime.now() + timedelta(hours=6)),
            reserve_price=47.20
        ),
        DeliveryPackage(
            id=1002, customer_id=892, origin_mfc=12,
            destination=(42.1, 38.9), size=2.1, priority=1,
            time_window=(datetime.now() + timedelta(hours=0.5), datetime.now() + timedelta(hours=4)),
            reserve_price=52.80
        )
    ]
    
    print(f"V-23 Location: {v23.location}")
    print(f"V-23 Available Capacity: {v23.capacity - v23.current_load}m³")
    print(f"MFC-7 Package: 2.8m³, reserve price: $47.20")
    print(f"MFC-12 Package: 2.1m³, reserve price: $52.80")
    
    # Create auction context
    context = {
        'current_time': datetime.now(),
        'market_factor': 1.0,
        'competitors': 2,
        'demand_factor': 1.1
    }
    
    # Run auctions
    print(f"\n🤝 RUNNING NEGOTIATION AUCTIONS...")
    
    auction_mechanism = AuctionMechanism()
    negotiation_results = []
    
    for i, package in enumerate(packages):
        print(f"\n--- Auction for Package {package.id} (from MFC-{package.origin_mfc}) ---")
        
        # Get bids from all agents
        all_agents = [v23, mfc7, mfc12, mfc18]
        
        # Collect bids
        bids = []
        for agent in all_agents:
            bid = agent.make_bid(package, context)
            if bid:
                bids.append((agent, bid))
                print(f"   {agent.type.value.upper()}-{agent.id}: ${bid.price:.2f} "
                      f"(confidence: {bid.confidence:.2f}, capacity: {bid.capacity:.1f})")
        
        if bids:
            # Determine winner
            bids.sort(key=lambda x: (x[1].price, x[1].confidence), reverse=True)
            winner, winning_bid = bids[0]
            
            print(f"   🏆 WINNER: {winner.type.upper()}-{winner.id} at ${winning_bid.price:.2f}")
            
            # Record result
            negotiation_results.append({
                'package_id': package.id,
                'winner': winner.type.value,
                'winner_id': winner.id,
                'price': winning_bid.price,
                'package_size': package.size
            })
            
            # Update winner state
            if isinstance(winner, VehicleAgent):
                winner.current_load += package.size
        else:
            print(f"   ❌ No valid bids received")
    
    # Calculate final results
    print(f"\n📊 NEGOTIATION SUMMARY:")
    total_revenue = sum(r['price'] for r in negotiation_results)
    total_volume = sum(r['package_size'] for r in negotiation_results)
    vehicle_utilization = v23.current_load / v23.capacity * 100
    
    print(f"   Total revenue: ${total_revenue:.2f}")
    print(f"   Total volume: {total_volume:.1f}m³")
    print(f"   V-23 utilization: {vehicle_utilization:.1f}%")
    print(f"   Negotiations completed: {len(negotiation_results)}")
    
    # Compare with problem statement expectations
    print(f"\n📋 COMPARISON WITH PROBLEM STATEMENT:")
    print(f"   Expected V-23 utilization: ~94.3%")
    print(f"   Actual V-23 utilization: {vehicle_utilization:.1f}%")
    print(f"   Expected on-time delivery: ~97.1%")
    print(f"   System efficiency: {'High' if vehicle_utilization > 85 else 'Medium' if vehicle_utilization > 70 else 'Low'}")
    
    if vehicle_utilization >= 85:
        print(f"   ✅ High utilization achieved!")
    elif vehicle_utilization >= 70:
        print(f"   ⚠️ Moderate utilization")
    else:
        print(f"   ❌ Low utilization")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'VehicleAgent' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Autonomous Self-Optimizing Ecosystem - Distributed Intelligence', fontsize=16, fontweight='bold')
    
    # 1. Convergence to Nash Equilibrium
    ax1 = axes[0, 0]
    if optimization_results['total_utility']:
        iterations = optimization_results['iterations']
        utilities = optimization_results['total_utility']
        allocations = optimization_results['packages_allocated']
        
        ax1.plot(iterations, utilities, 'b-', linewidth=2, marker='o', markersize=4, label='Total Utility')
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Total System Utility', color='b')
        ax1.tick_params(axis='y', labelcolor='b')
        ax1.grid(True, alpha=0.3)
        
        # Add allocation as secondary axis
        ax1_twin = ax1.twinx()
        ax1_twin.plot(iterations, allocations, 'r--', linewidth=2, marker='s', markersize=3, label='Packages Allocated')
        ax1_twin.set_ylabel('Packages Allocated', color='r')
        ax1_twin.tick_params(axis='y', labelcolor='r')
        
        # Mark equilibrium point
        if ecosystem.nash_equilibrium_achieved:
            eq_iteration = ecosystem.equilibrium_iterations
            eq_utility = utilities[eq_iteration - 1]
            ax1.axvline(x=eq_iteration, color='green', linestyle=':', alpha=0.7, label='Nash Equilibrium')
            ax1.annotate(f'Equilibrium\n(iteration {eq_iteration})', 
                        xy=(eq_iteration, eq_utility), 
                        xytext=(eq_iteration + 1, eq_utility * 1.1),
                        arrowprops=dict(arrowstyle='->', color='green'),
                        fontsize=9, ha='center')
        
        ax1.set_title('Convergence to Nash Equilibrium')
        ax1.legend(loc='upper left')
    
    # 2. Agent Performance Distribution
    ax2 = axes[0, 1]
    if optimization_results['auction_results'] and optimization_results['auction_results'][-1]:
        final_auctions = optimization_results['auction_results'][-1]
        
        # Group by agent type
        vehicle_utilities = [a['utility'] for a in final_auctions if a['winner_type'] == 'vehicle']
        
        # Create violin plot style distribution
        if vehicle_utilities:
            parts = ax2.violinplot([vehicle_utilities], positions=[1], widths=0.3)
            parts['bodies'][0].set_facecolor('lightblue')
            parts['bodies'][0].set_alpha(0.7)
            
            # Add statistics
            mean_utility = np.mean(vehicle_utilities)
            std_utility = np.std(vehicle_utilities)
            
            ax2.axhline(y=mean_utility, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_utility:.2f}')
            ax2.fill_between([0.7, 1.3], mean_utility - std_utility, mean_utility + std_utility, 
                             alpha=0.2, color='red', label=f'±1σ: {std_utility:.2f}')
        
        ax2.set_xlim(0.5, 1.5)
        ax2.set_ylabel('Utility per Agent')
        ax2.set_title('Vehicle Agent Utility Distribution')
        ax2.set_xticks([1])
        ax2.set_xticklabels(['Vehicle Agents'])
        ax2.grid(True, alpha=0.3)
        ax2.legend()
    
    # 3. Auction Price Analysis
    ax3 = axes[1, 0]
    if optimization_results['auction_results']:
        # Collect all auction prices across iterations
        all_prices = []
        iteration_labels = []
        
        for i, iteration_auctions in enumerate(optimization_results['auction_results']):
            if iteration_auctions:
                prices = [a['price'] for a in iteration_auctions]
                all_prices.extend(prices)
                iteration_labels.extend([f'Iter {i+1}'] * len(prices))
        
        if all_prices:
            # Create box plot by iteration
            unique_iterations = list(set(iteration_labels))
            price_by_iteration = []
            
            for iteration in unique_iterations:
                iter_prices = [all_prices[i] for i, label in enumerate(iteration_labels) if label == iteration]
                price_by_iteration.append(iter_prices)
            
            bp = ax3.boxplot(price_by_iteration, patch_artist=True, labels=unique_iterations)
            
            # Color the boxes
            colors = plt.cm.Set3(np.linspace(0, 1, len(bp['boxes'])))
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            ax3.set_xlabel('Optimization Iteration')
            ax3.set_ylabel('Auction Price ($)')
            ax3.set_title('Auction Price Evolution')
            ax3.grid(True, alpha=0.3)
    
    # 4. Geographic Agent Distribution
    ax4 = axes[1, 1]
    
    # Plot vehicles
    for vehicle in ecosystem.vehicles:
        color = {'van': 'blue', 'bike': 'green', 'drone': 'red'}.get(vehicle.vehicle_type, 'gray')
        ax4.plot(vehicle.location[0], vehicle.location[1], 'o', 
                markersize=8, color=color, alpha=0.7)
        ax4.text(vehicle.location[0] + 1, vehicle.location[1] + 1, 
                f'V-{vehicle.id}', fontsize=8)
    
    # Plot MFCs
    for mfc in ecosystem.mfcs:
        ax4.plot(mfc.location[0], mfc.location[1], 's', 
                markersize=12, color='orange', alpha=0.8)
        ax4.text(mfc.location[0] + 1, mfc.location[1] + 1, 
                f'MFC-{mfc.id}', fontsize=8, fontweight='bold')
    
    # Plot package destinations
    for package in ecosystem.packages[:10]:  # Show first 10
        ax4.plot(package.destination[0], package.destination[1], '^', 
                markersize=6, color='purple', alpha=0.5)
    
    # Add legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
                   markersize=8, label='Van'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green', 
                   markersize=8, label='Bike'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                   markersize=8, label='Drone'),
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
                   markersize=10, label='MFC'),
        plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='purple', 
                   markersize=6, label='Package')
    ]
    ax4.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    ax4.set_xlim(0, ecosystem.city_size[0])
    ax4.set_ylim(0, ecosystem.city_size[1])
    ax4.set_xlabel('X Coordinate (km)')
    ax4.set_ylabel('Y Coordinate (km)')
    ax4.set_title('Geographic Agent Distribution')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Autonomous Ecosystem visualization complete!")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'optimization_results' is not defined...]

In [ ]:
try:
    # System analysis and performance assessment
    print("=" * 60)
    print('AUTONOMOUS ECOSYSTEM PERFORMANCE ANALYSIS')
    print("=" * 60)
    
    # Nash Equilibrium Analysis
    print(f"🎯 NASH EQUILIBRIUM ANALYSIS:")
    if ecosystem.nash_equilibrium_achieved:
        print(f"   ✅ Nash equilibrium achieved: Yes")
        print(f"   🔄 Convergence iteration: {ecosystem.equilibrium_iterations}")
        print(f"   ⚡ Convergence speed: {'Fast' if ecosystem.equilibrium_iterations <= 3 else 'Medium' if ecosystem.equilibrium_iterations <= 6 else 'Slow'}")
    else:
        print(f"   ⚠️ Nash equilibrium not fully achieved within iterations")
        print(f"   📈 Trend: {'Improving' if len(optimization_results['total_utility']) > 1 and optimization_results['total_utility'][-1] > optimization_results['total_utility'][0] else 'Stable'}")
    
    # System Efficiency Analysis
    if optimization_results['total_utility']:
        final_utility = optimization_results['total_utility'][-1]
        initial_utility = optimization_results['total_utility'][0]
        utility_improvement = ((final_utility - initial_utility) / initial_utility) * 100 if initial_utility > 0 else 0
        
        print(f"\n📈 SYSTEM EFFICIENCY:")
        print(f"   Initial utility: {initial_utility:.2f}")
        print(f"   Final utility: {final_utility:.2f}")
        print(f"   Improvement: {utility_improvement:.1f}%")
        print(f"   Efficiency: {'High' if utility_improvement > 20 else 'Medium' if utility_improvement > 10 else 'Low'}")
    
    # Agent Behavior Analysis
    print(f"\n🤖 AGENT BEHAVIOR ANALYSIS:")
    
    # Vehicle agent analysis
    vehicle_types = {}
    for vehicle in ecosystem.vehicles:
        vehicle_types[vehicle.vehicle_type] = vehicle_types.get(vehicle.vehicle_type, 0) + 1
    
    print(f"   Vehicle fleet composition:")
    for v_type, count in vehicle_types.items():
        print(f"     - {v_type}: {count} agents")
    
    # Auction analysis
    total_auctions = len(ecosystem.auction.auction_history)
    if total_auctions > 0:
        avg_auction_price = np.mean([a['winning_price'] for a in ecosystem.auction.auction_history])
        vehicle_win_rate = len([a for a in ecosystem.auction.auction_history if a['winner_type'] == 'vehicle']) / total_auctions * 100
        
        print(f"\n💰 AUCTION MARKET ANALYSIS:")
        print(f"   Total auctions: {total_auctions}")
        print(f"   Average price: ${avg_auction_price:.2f}")
        print(f"   Vehicle win rate: {vehicle_win_rate:.1f}%")
        print(f"   Market activity: {'High' if total_auctions > 50 else 'Medium' if total_auctions > 25 else 'Low'}")
    
    # Distributed Decision Quality
    print(f"\n🔄 DISTRIBUTED DECISION QUALITY:")
    if optimization_results['packages_allocated']:
        avg_allocation_rate = np.mean(optimization_results['packages_allocated']) / len(ecosystem.packages) * 100
        final_allocation_rate = optimization_results['packages_allocated'][-1] / len(ecosystem.packages) * 100
        
        print(f"   Average allocation rate: {avg_allocation_rate:.1f}%")
        print(f"   Final allocation rate: {final_allocation_rate:.1f}%")
        print(f"   Decision consistency: {'High' if np.std(optimization_results['packages_allocated']) < 2 else 'Medium' if np.std(optimization_results['packages_allocated']) < 4 else 'Low'}")
    
    # Self-Organization Assessment
    print(f"\n🌊 SELF-ORGANIZATION ASSESSMENT:")
    
    # Calculate agent interaction diversity
    if ecosystem.auction.auction_history:
        unique_winners = len(set(a['winner_id'] for a in ecosystem.auction.auction_history))
        total_agents = len(ecosystem.vehicles) + len(ecosystem.mfcs)
        participation_rate = unique_winners / total_agents * 100
        
        print(f"   Agent participation: {participation_rate:.1f}%")
        print(f"   Unique winners: {unique_winners}/{total_agents}")
        print(f"   Decentralization: {'High' if participation_rate > 80 else 'Medium' if participation_rate > 60 else 'Low'}")
    
    # Emergent Properties
    print(f"\n✨ EMERGENT PROPERTIES:")
    print(f"   Nash equilibrium: {'Achieved' if ecosystem.nash_equilibrium_achieved else 'Emerging'}")
    print(f"   Market clearing: {'Efficient' if vehicle_win_rate > 70 else 'Developing'}")
    print(f"   Load balancing: {'Good' if final_allocation_rate > 80 else 'Needs improvement'}")
    print(f"   System resilience: {'High' if utility_improvement > 15 else 'Medium'}")
    
    # Comparison with expected results
    print(f"\n📋 COMPARISON WITH EXPECTED RESULTS:")
    print(f"   Expected vehicle utilization: ~94.3%")
    actual_utilization = vehicle_utilization if 'vehicle_utilization' in locals() else 0
    print(f"   Actual vehicle utilization: {actual_utilization:.1f}%")
    print(f"   Expected on-time delivery: ~97.1%")
    print(f"   System performance: {'Matches expectations' if actual_utilization > 85 else 'Below expectations'}")
    
    # Final Assessment
    print(f"\n🎯 FINAL ECOSYSTEM ASSESSMENT:")
    
    assessment_score = 0
    max_score = 5
    
    if ecosystem.nash_equilibrium_achieved:
        assessment_score += 1
        print(f"   ✅ Nash equilibrium achieved")
    
    if utility_improvement > 10:
        assessment_score += 1
        print(f"   ✅ System improvement demonstrated")
    
    if final_allocation_rate > 75:
        assessment_score += 1
        print(f"   ✅ High allocation efficiency")
    
    if participation_rate > 70:
        assessment_score += 1
        print(f"   ✅ Good agent participation")
    
    if avg_auction_price > 30:
        assessment_score += 1
        print(f"   ✅ Healthy market prices")
    
    final_score = (assessment_score / max_score) * 100
    print(f"\n🏆 OVERALL ECOSYSTEM SCORE: {final_score:.0f}%")
    
    if final_score >= 80:
        assessment = "Excellent - Autonomous ecosystem functioning optimally"
    elif final_score >= 60:
        assessment = "Good - Ecosystem showing strong self-organization"
    elif final_score >= 40:
        assessment = "Fair - Ecosystem functional but needs optimization"
    else:
        assessment = "Needs Improvement - Ecosystem requires refinement"
    
    print(f"   {assessment}")
    print(f"   🤖 Distributed intelligence enables {total_auctions:,} autonomous decisions")
    print(f"   🎯 Nash equilibrium provides stable system configuration")
    print(f"   🔄 Self-organization emerges without centralized control")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ecosystem' is not defined...]

### Why this Tier exists vs Tiers 1-5
The Autonomous Self-Optimizing Ecosystem provides revolutionary capabilities beyond previous approaches:
- **Distributed intelligence**: Agent-based decision-making vs centralized optimization
- **Self-organization**: Emergent system behavior vs pre-programmed solutions
- **Game-theoretic optimization**: Nash equilibrium vs single-objective optimization
- **Autonomous negotiation**: Market-based resource allocation vs algorithmic assignment

### Pros / Cons vs Tiers 1-5
**Pros:**
- True autonomy and self-organization
- Scalable through distributed architecture
- Resilient to single points of failure
- Adaptable to changing conditions
- Market-based efficiency optimization
- No centralized control required
- Emergent intelligent behavior
- Natural load balancing

**Cons:**
- Complex agent behavior modeling
- Difficult to predict exact outcomes
- Requires sophisticated mechanism design
- Potential for suboptimal equilibria
- Communication overhead between agents
- Challenging to debug and maintain
- May require significant computational resources
- Equilibrium convergence not guaranteed

### When to use this Tier
- Large-scale distributed delivery networks
- When system autonomy is critical
- Environments requiring high resilience
- Multi-stakeholder operations
- When centralized control is impractical
- For future-proof logistics systems
- When market-based allocation is preferred
- Systems requiring continuous adaptation